# Beam Search for Agents | Advanced Planning & Search

In [1]:
from langchain_openai import ChatOpenAI
from typing import List
from dataclasses import dataclass
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
import json
import re

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
BEAM_WIDTH = 3
MAX_STEPS = 3

@dataclass
class Beam:
    content: str
    score: float
    steps: int

def expand_beam(beam: Beam, problem: str) -> List[Beam]:
    """Generate candidate expansions for a beam."""
    response = model.invoke(
        f"You are solving: {problem}\n\n"
        f"Current partial solution:\n{beam.content}\n\n"
        f"Generate 3 different next steps. For each, provide the step and a quality score (0-10).\n"
        f"Return JSON: [{{\"step\": \"...\", \"score\": 8.5}}]"
    )
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", response.content).strip()
    try:
        expansions = json.loads(cleaned)
    except json.JSONDecodeError:
        expansions = [{"step": response.content, "score": 5.0}]
    result = []
    for exp in expansions:
        if isinstance(exp, str):
            exp = {"step": exp, "score": 5.0}
        new_content = f"{beam.content}\n{beam.steps + 1}. {exp.get('step', '')}"
        result.append(Beam(content=new_content, score=float(exp.get("score", 5)), steps=beam.steps + 1))
    return result

def beam_search(problem: str) -> str:
    # Initialize beams
    beams = [Beam(content="Solution steps:", score=5.0, steps=0)]
    greedy_beam = beams[0]  # track top-1 only for greedy comparison

    for step in range(MAX_STEPS):
        all_candidates = []
        for beam in beams:
            expansions = expand_beam(beam, problem)
            all_candidates.extend(expansions)
        # Keep top-K
        all_candidates.sort(key=lambda b: b.score, reverse=True)
        beams = all_candidates[:BEAM_WIDTH]
        # Greedy baseline: expand only the single best beam from previous step
        greedy_expansions = expand_beam(greedy_beam, problem)
        greedy_beam = max(greedy_expansions, key=lambda b: b.score)
        unique_approaches = len(set(b.content.split('\n')[-1] for b in beams))
        print(f"Step {step + 1}: {len(all_candidates)} candidates -> kept {len(beams)}, {unique_approaches} unique approaches")

    best = max(beams, key=lambda b: b.score)
    # --- Beam vs Greedy comparison ---
    found_better = "Yes" if best.score > greedy_beam.score else "No"
    print(f"\nBeam search: {best.score:.2f} | Greedy (top-1 only): {greedy_beam.score:.2f} | Beam found better path: {found_better}")
    return f"Best solution (score: {best.score}):\n{best.content}"

In [5]:
result = beam_search("Design an authentication system supporting OAuth2, SAML, and API keys")
print(result)

Step 1: 1 candidates -> kept 1, 1 unique approaches
Step 2: 3 candidates -> kept 3, 3 unique approaches
Step 3: 9 candidates -> kept 3, 3 unique approaches

Beam search: 9.10 | Greedy (top-1 only): 9.00 | Beam found better path: Yes
Best solution (score: 9.1):
Solution steps:
1. Here's a set of three different next steps with their quality scores:

```json
[
    {
        "step": "Integrate OAuth2 framework into the authentication system by supporting authorization flows like authorization code, implicit, and client credentials.",
        "score": 8.5
    },
    {
        "step": "Implement SAML 2.0 by setting up a SAML identity provider (IdP) and configuring your service as a SAML service provider (SP) to handle SSO and single logout (SLO) requests.",
        "score": 9.0
    },
    {
        "step": "Design and build an API key management module, including generating, revoking, and verifying API keys, while ensuring secure storage and transmission.",
        "score": 8.0
    }
]
```
